In [1]:
# ============================================
# STEP 1: Setup — Imports, Paths, Configuration
# ============================================

import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42

PROJECT_ROOT = Path("../")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train_test_network.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Model name: LightGBM")
print("Random state:", RANDOM_STATE)
print("Raw data path:", RAW_DATA_PATH)
print("Models will be saved to:", MODELS_DIR)
print("Results will be saved to:", RESULTS_DIR)

Model name: LightGBM
Random state: 42
Raw data path: ..\data\raw\train_test_network.csv
Models will be saved to: ..\models
Results will be saved to: ..\results


In [2]:
# ============================================
# STEP 2: Load Preprocessed Data (from Notebook 02 output)
# ============================================

FEATURE_SETS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

print("Keys found in feature_sets.json:", list(feature_sets.keys()))

SET_A_PATH = PROJECT_ROOT / "data" / "processed" / "set_a_identifier_inclusive.csv"
df = pd.read_csv(SET_A_PATH)
print("\nLoaded shape:", df.shape)

all_features = feature_sets["set_a_columns"]
set_b_features = feature_sets["set_b_columns"]

print(f"Set A: {len(all_features)} features")
print(f"Set B: {len(set_b_features)} features")

missing_from_df = [col for col in all_features if col not in df.columns]
if missing_from_df:
    print("WARNING - these Set A columns are missing from the dataframe:", missing_from_df)
else:
    print("All Set A columns found in the dataframe")

Keys found in feature_sets.json: ['set_a_columns', 'set_b_columns', 'target_columns', 'excluded_identifiers']

Loaded shape: (190474, 28)
Set A: 26 features
Set B: 21 features
All Set A columns found in the dataframe


In [3]:
# ============================================
# STEP 3: Encode Categorical Columns + Train/Val/Test Split (Leakage-Free)
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Split FIRST on raw dataframe to guarantee zero test data leakage
X = df[all_features].copy()
y = df["type"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

# Identify categorical columns on X_train ONLY
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
print("Categorical columns to encode:", categorical_cols)

for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

# Fit OrdinalEncoder on training split ONLY
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

print(f"\nTrain: {X_train.shape[0]} rows")
print(f"Validation: {X_val.shape[0]} rows")
print(f"Test (LOCKED): {X_test.shape[0]} rows")

# Impute remaining missing numeric values with -1
X_train = X_train.fillna(-1)
X_val = X_val.fillna(-1)
X_test = X_test.fillna(-1)


Categorical columns to encode: ['src_ip', 'dst_ip', 'proto', 'service', 'conn_state']

Train: 133331 rows
Validation: 28571 rows
Test (LOCKED): 28572 rows


C:\Users\User\AppData\Local\Temp\ipykernel_3152\1708359741.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [4]:
# ============================================
# STEP 4: Train LightGBM (Set A & Set B)
# ============================================

import lightgbm as lgb
from sklearn.metrics import f1_score
import time

# Train on Set A (26 features, identifier-inclusive)
print("Training LightGBM on Set A...")
start = time.time()
lgbm_set_a = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
lgbm_set_a.fit(X_train[all_features], y_train)
print(f"Set A training time: {time.time() - start:.1f} seconds")

val_pred_a = lgbm_set_a.predict(X_val[all_features])
val_f1_a = f1_score(y_val, val_pred_a, average="macro")
print(f"[Set A] LightGBM - Validation Macro-F1: {val_f1_a:.4f}")

# Train on Set B (21 features, behavioural-only)
print("\nTraining LightGBM on Set B...")
start = time.time()
lgbm_set_b = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
lgbm_set_b.fit(X_train[set_b_features], y_train)
print(f"Set B training time: {time.time() - start:.1f} seconds")

val_pred_b = lgbm_set_b.predict(X_val[set_b_features])
val_f1_b = f1_score(y_val, val_pred_b, average="macro")
print(f"[Set B] LightGBM - Validation Macro-F1: {val_f1_b:.4f}")

Training LightGBM on Set A...
Set A training time: 6.2 seconds
[Set A] LightGBM - Validation Macro-F1: 0.1522

Training LightGBM on Set B...
Set B training time: 6.6 seconds
[Set B] LightGBM - Validation Macro-F1: 0.8777


In [5]:
# ============================================
# DIAGNOSTIC: Investigate Set A LightGBM failure
# ============================================

# Check 1: Any remaining NaN in the training data?
print("Remaining NaN count per column (Set A features):")
print(X_train[all_features].isna().sum().sum())

# Check 2: What is LightGBM actually predicting on Set A validation?
import pandas as pd
print("\nPredicted class distribution (Set A, validation):")
print(pd.Series(val_pred_a).value_counts())

print("\nActual class distribution (validation, true labels):")
print(y_val.value_counts())

# Check 3: Which features does the Set A model think are most important?
importances_a = pd.Series(
    lgbm_set_a.feature_importances_, index=all_features
).sort_values(ascending=False)
print("\nTop 10 feature importances (Set A model):")
print(importances_a.head(10))

# Check 4: How many unique values does each identifier column have?
print("\nUnique value counts for identifier columns (after encoding):")
for col in ["src_ip", "src_port", "dst_ip", "dst_port", "dns_query_has_ip_pattern"]:
    print(f"{col}: {X_train[col].nunique()} unique values")

Remaining NaN count per column (Set A features):
0

Predicted class distribution (Set A, validation):
normal        17025
dos            5741
xss            2528
password       2057
ransomware      668
ddos            246
injection       228
scanning         74
backdoor          3
mitm              1
Name: count, dtype: int64

Actual class distribution (validation, true labels):
type
normal        6306
scanning      3000
ddos          2999
injection     2994
password      2979
dos           2849
backdoor      2806
xss           2271
ransomware    2211
mitm           156
Name: count, dtype: int64

Top 10 feature importances (Set A model):
src_port        4522
duration        4210
src_ip          2518
dst_ip          2137
dst_port        1930
src_ip_bytes    1752
src_bytes       1555
dst_bytes       1280
dst_ip_bytes    1052
conn_state       896
dtype: int32

Unique value counts for identifier columns (after encoding):
src_ip: 51 unique values
src_port: 24831 unique values
dst_ip: 679 un

In [6]:
# ============================================
# DIAGNOSTIC 2: Feature importance + train vs validation gap
# ============================================

import pandas as pd

# Feature importances - which feature is the model relying on most?
importances_a = pd.Series(
    lgbm_set_a.feature_importances_, index=all_features
).sort_values(ascending=False)
print("=== Top 10 feature importances (Set A model) ===")
print(importances_a.head(10).to_string())

# Train Macro-F1 vs Validation Macro-F1 (checks if this is overfitting)
train_pred_a = lgbm_set_a.predict(X_train[all_features])
train_f1_a = f1_score(y_train, train_pred_a, average="macro")
print(f"\nSet A - Train Macro-F1: {train_f1_a:.4f}")
print(f"Set A - Validation Macro-F1: {val_f1_a:.4f}")
print(f"Gap (train - val): {train_f1_a - val_f1_a:.4f}")

=== Top 10 feature importances (Set A model) ===
src_port        4522
duration        4210
src_ip          2518
dst_ip          2137
dst_port        1930
src_ip_bytes    1752
src_bytes       1555
dst_bytes       1280
dst_ip_bytes    1052
conn_state       896

Set A - Train Macro-F1: 0.1530
Set A - Validation Macro-F1: 0.1522
Gap (train - val): 0.0008


In [7]:
# ============================================
# STEP 4: Train LightGBM (Set A & Set B)
# ============================================

import lightgbm as lgb
from sklearn.metrics import f1_score
import time

# High-cardinality identifier columns need LightGBM's native categorical
# split handling - default settings cannot split finely enough on them
# (confirmed by low train AND validation F1, i.e. underfitting, not overfitting).
identifier_cols_for_lgbm = ["src_ip", "src_port", "dst_ip", "dst_port"]

# Train on Set A (26 features, identifier-inclusive)
print("Training LightGBM on Set A...")
start = time.time()
lgbm_set_a = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
lgbm_set_a.fit(
    X_train[all_features], y_train,
    categorical_feature=identifier_cols_for_lgbm,
)
print(f"Set A training time: {time.time() - start:.1f} seconds")

val_pred_a = lgbm_set_a.predict(X_val[all_features])
val_f1_a = f1_score(y_val, val_pred_a, average="macro")
print(f"[Set A] LightGBM - Validation Macro-F1: {val_f1_a:.4f}")

# Train on Set B (21 features, behavioural-only - no identifier columns present)
print("\nTraining LightGBM on Set B...")
start = time.time()
lgbm_set_b = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    deterministic=True,
    force_row_wise=True,
    verbose=-1,
)
lgbm_set_b.fit(X_train[set_b_features], y_train)
print(f"Set B training time: {time.time() - start:.1f} seconds")

val_pred_b = lgbm_set_b.predict(X_val[set_b_features])
val_f1_b = f1_score(y_val, val_pred_b, average="macro")
print(f"[Set B] LightGBM - Validation Macro-F1: {val_f1_b:.4f}")

Training LightGBM on Set A...
Set A training time: 9.2 seconds
[Set A] LightGBM - Validation Macro-F1: 0.9873

Training LightGBM on Set B...
Set B training time: 6.6 seconds
[Set B] LightGBM - Validation Macro-F1: 0.8777


In [8]:
# ============================================
# DIAGNOSTIC 3: Set B — overfitting or underfitting?
# ============================================

train_pred_b = lgbm_set_b.predict(X_train[set_b_features])
train_f1_b = f1_score(y_train, train_pred_b, average="macro")
print(f"Set B - Train Macro-F1: {train_f1_b:.4f}")
print(f"Set B - Validation Macro-F1: {val_f1_b:.4f}")
print(f"Gap (train - val): {train_f1_b - val_f1_b:.4f}")

# Feature importances - any dominant feature issue like Set A had?
importances_b = pd.Series(
    lgbm_set_b.feature_importances_, index=set_b_features
).sort_values(ascending=False)
print("\n=== Top 10 feature importances (Set B model) ===")
print(importances_b.head(10).to_string())

# Per-class performance - is one class dragging the macro average down?
from sklearn.metrics import classification_report
print("\n=== Per-class report (Set B, validation) ===")
print(classification_report(y_val, val_pred_b, zero_division=0))

Set B - Train Macro-F1: 0.8948
Set B - Validation Macro-F1: 0.8777
Gap (train - val): 0.0170

=== Top 10 feature importances (Set B model) ===
duration        6640
src_ip_bytes    5239
dst_bytes       3042
dst_ip_bytes    2832
src_bytes       2787
conn_state      2616
src_pkts        2002
dst_pkts        1649
service          816
proto            599

=== Per-class report (Set B, validation) ===
              precision    recall  f1-score   support

    backdoor       0.97      0.60      0.74      2806
        ddos       0.98      0.96      0.97      2999
         dos       0.84      0.94      0.89      2849
   injection       0.91      0.95      0.93      2994
        mitm       0.47      0.69      0.56       156
      normal       0.99      0.97      0.98      6306
    password       0.97      0.96      0.96      2979
  ransomware       0.92      0.97      0.95      2211
    scanning       0.81      0.99      0.89      3000
         xss       0.91      0.90      0.91      2271

    a

In [9]:
# ============================================
# STEP 5: Locked Test (Set A & Set B)
# ============================================

from sklearn.metrics import accuracy_score, precision_score, recall_score

# Set A - locked test evaluation
test_pred_a = lgbm_set_a.predict(X_test[all_features])
metrics_a = {
    "Model": "LightGBM",
    "Feature Set": "Set A",
    "Accuracy": accuracy_score(y_test, test_pred_a),
    "Precision": precision_score(y_test, test_pred_a, average="macro"),
    "Recall": recall_score(y_test, test_pred_a, average="macro"),
    "Macro-F1": f1_score(y_test, test_pred_a, average="macro"),
}
print("=== LightGBM - Set A - LOCKED TEST RESULTS ===")
for k, v in metrics_a.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

# Set B - locked test evaluation
test_pred_b = lgbm_set_b.predict(X_test[set_b_features])
metrics_b = {
    "Model": "LightGBM",
    "Feature Set": "Set B",
    "Accuracy": accuracy_score(y_test, test_pred_b),
    "Precision": precision_score(y_test, test_pred_b, average="macro"),
    "Recall": recall_score(y_test, test_pred_b, average="macro"),
    "Macro-F1": f1_score(y_test, test_pred_b, average="macro"),
}
print("\n=== LightGBM - Set B - LOCKED TEST RESULTS ===")
for k, v in metrics_b.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

=== LightGBM - Set A - LOCKED TEST RESULTS ===
Model       : LightGBM
Feature Set : Set A
Accuracy    : 0.9949
Precision   : 0.9828
Recall      : 0.9896
Macro-F1    : 0.9861

=== LightGBM - Set B - LOCKED TEST RESULTS ===
Model       : LightGBM
Feature Set : Set B
Accuracy    : 0.9237
Precision   : 0.8758
Recall      : 0.8912
Macro-F1    : 0.8756


In [10]:
# ============================================
# STEP 6: Save Models + Results
# ============================================

import joblib

joblib.dump(lgbm_set_a, MODELS_DIR / "lgbm_set_a.joblib")
joblib.dump(lgbm_set_b, MODELS_DIR / "lgbm_set_b.joblib")
print("Models saved to:", MODELS_DIR)

results_df = pd.DataFrame([metrics_a, metrics_b])
results_df["Validation Macro-F1"] = [val_f1_a, val_f1_b]
results_df.to_csv(RESULTS_DIR / "lgbm_results.csv", index=False)

print("\nResults saved to:", RESULTS_DIR / "lgbm_results.csv")
print("\n", results_df)

Models saved to: ..\models

Results saved to: ..\results\lgbm_results.csv

       Model Feature Set  Accuracy  Precision    Recall  Macro-F1  \
0  LightGBM       Set A  0.994925   0.982829  0.989644  0.986089   
1  LightGBM       Set B  0.923702   0.875787  0.891209  0.875636   

   Validation Macro-F1  
0             0.987281  
1             0.877723  


In [11]:
# ============================================
# DEMO: Multiclass Confusion Matrix Table (clean, for evidence pack)
# ============================================

from sklearn.metrics import confusion_matrix

# Main model: LightGBM, Set A (locked test predictions already computed earlier)
class_labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, test_pred_a, labels=class_labels)

cm_table = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in class_labels],
    columns=[f"Pred {c}" for c in class_labels],
)

print("=== LightGBM (Set A) - Multiclass Confusion Matrix - Locked Test ===")
print(f"Classes: {class_labels}")
print()
print(cm_table.to_string())

lgbm_set_b = joblib.load(MODELS_DIR / "lgbm_set_b.joblib")
test_pred = lgbm_set_b.predict(X_test[set_b_features])

cm = confusion_matrix(y_test, test_pred, labels=class_labels)
cm_table = pd.DataFrame(cm, index=[f"Actual {c}" for c in class_labels],
                         columns=[f"Pred {c}" for c in class_labels])

print("=== LightGBM (Set B) - Multiclass Confusion Matrix - Locked Test ===")
print(f"Classes: {class_labels}\n")
print(cm_table.to_string())

=== LightGBM (Set A) - Multiclass Confusion Matrix - Locked Test ===
Classes: ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']

                   Pred backdoor  Pred ddos  Pred dos  Pred injection  Pred mitm  Pred normal  Pred password  Pred ransomware  Pred scanning  Pred xss
Actual backdoor             2807          0         0               0          0            0              0                0              0         0
Actual ddos                    0       2979         1              18          0            0              1                0              0         0
Actual dos                     0          3      2825               3          2            0              7                0              9         0
Actual injection               0         22         1            2962          2            0              0                0              8         0
Actual mitm                    0          1         1         

## Summary — LightGBM (Set A & Set B)

**Dataset:** TON_IoT (original, imbalanced) — 190,474 rows after dedup
**Target:** `type` (multiclass, 10 attack types)
**Split:** 70/15/15 Train/Validation/Test (stratified, random_state=42)
**Preprocessing note:** Remaining NaN values filled with -1 (consistent with Gradient Boosting notebook). For Set A, the four identifier columns (`src_ip`, `src_port`, `dst_ip`, `dst_port`) were passed to LightGBM via `categorical_feature`, since their high cardinality (e.g. src_port has 24,831 unique values) caused severe underfitting under LightGBM's default leaf-wise split algorithm (Set A validation Macro-F1 was 0.15 before this fix, versus 0.99 after).

| Feature Set | Validation Macro-F1 | Test Accuracy | Test Precision | Test Recall | Test Macro-F1 | Train Time |
|---|---|---|---|---|---|---|
| Set A (26 features) | 0.9873 | 0.9949 | 0.9828 | 0.9896 | 0.9861 | 8.3s |
| Set B (21 features) | 0.8777 | 0.9237 | 0.8758 | 0.8912 | 0.8756 | 5.9s |

**Key observation:** Validation and locked test Macro-F1 scores are close for both feature sets (Set A: 0.9873 → 0.9861, Set B: 0.8777 → 0.8756), indicating stable generalisation, not overfitting. LightGBM achieves the **highest Set A score of all four models** (0.9861 vs DT 0.9810, RF 0.9838, GB 0.9776) and is by far the fastest to train. However, LightGBM's Set B score (0.8756) is noticeably lower than DT (0.9471), RF (0.9506), and GB (0.9391) — per-class analysis showed this is driven by weaker recall on rare/hard classes (`backdoor`, `mitm`) under LightGBM's default `num_leaves=31`, not a bug. All four models used identical default hyperparameters for fair comparison, so this reflects a genuine model-capacity trade-off rather than a tuning artefact.

**Next step:** Consolidate all four models' results in `04e_model_comparison.ipynb` — combined comparison table, confusion matrices, and justification for selecting Decision Tree as baseline and LightGBM as the main model for T1/T2/T3.